# Feature Engineering Techniques – Extended Solution

**Domain:** Banking / Credit Risk  
**Extended with:** domain features, scaling, binning, encodings, interactions, missing indicators, downstream check, simulation, audience notes, flowchart

![Flowchart](feature_engineering_flowchart.png)


## Goals
Turn raw applicant attributes into a rich, well-behaved feature set that can be fed to a PD / credit-scoring model.


## 0. Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.preprocessing import StandardScaler, MinMaxScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False

rng = np.random.default_rng(42)
print("Libraries imported. sklearn available:", SKLEARN_AVAILABLE)


## 1. Synthetic Credit Dataset
We create a small but realistic portfolio so every step is fully reproducible.


In [ ]:
n = 40
income = rng.normal(65000, 22000, n).clip(18000, 160000)
debt = income * rng.uniform(0.15, 1.4, n)
age = rng.integers(22, 68, n)
employment = rng.choice(['employed', 'self-employed', 'unemployed'], n, p=[0.65, 0.25, 0.10])
credit_history = rng.uniform(0.5, 25, n).round(1)
credit_limit = rng.uniform(2000, 35000, n)
revolving = credit_limit * rng.uniform(0.05, 1.3, n)
# simple default probability driven by DTI & utilization
dti_raw = debt / income
util_raw = revolving / credit_limit
logit = -3.5 + 2.8 * dti_raw + 1.6 * util_raw - 0.03 * age
prob = 1 / (1 + np.exp(-logit))
default = (rng.random(n) < prob).astype(int)

df = pd.DataFrame({
    'income': income.round(0),
    'debt': debt.round(0),
    'age': age,
    'employment': employment,
    'credit_history_years': credit_history,
    'revolving_balance': revolving.round(0),
    'credit_limit': credit_limit.round(0),
    'default': default
})
# inject a few missing values
df.loc[rng.choice(n, 3, replace=False), 'credit_limit'] = np.nan
df.loc[rng.choice(n, 2, replace=False), 'employment'] = np.nan

print(df.head(8))
print("\nShape:", df.shape)
print("Missing:\n", df.isna().sum())
print("Default rate: {:.1%}".format(df['default'].mean()))


## 2. Domain-Derived Features


In [ ]:
df['dti'] = df['debt'] / df['income']
df['utilization'] = df['revolving_balance'] / df['credit_limit']
df['log_income'] = np.log1p(df['income'])
df['credit_age'] = df['credit_history_years']   # already useful as-is

print(df[['income','debt','dti','utilization','log_income']].describe().round(3))


## 3. Numerical Transforms
### Log already done. Now scaling and binning.


In [ ]:
# --- Scaling ---
# Manual standardisation (works without sklearn)
def standard_scale(s):
    return (s - s.mean()) / s.std(ddof=0)

def minmax_scale(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

df['dti_std'] = standard_scale(df['dti'].fillna(df['dti'].median()))
df['dti_mm']  = minmax_scale(df['dti'].fillna(df['dti'].median()))
df['age_std'] = standard_scale(df['age'])

# --- Binning ---
df['income_bin'] = pd.qcut(df['income'], q=4, labels=['Q1','Q2','Q3','Q4'])
df['dti_bin']    = pd.cut(df['dti'], bins=[0, 0.3, 0.5, 0.8, np.inf],
                          labels=['low','med','high','very_high'])

print("Income bins:\n", df['income_bin'].value_counts().sort_index())
print("\nDTI bins vs default rate:")
print(df.groupby('dti_bin', observed=True)['default'].agg(['count','mean']).round(3))


In [ ]:
# Visualise scaling effect
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].hist(df['dti'].dropna(), bins=12, color='steelblue', edgecolor='k')
axes[0].set_title('Raw DTI')
axes[1].hist(df['dti_std'].dropna(), bins=12, color='seagreen', edgecolor='k')
axes[1].set_title('Standard-scaled DTI')
axes[2].hist(df['dti_mm'].dropna(), bins=12, color='darkorange', edgecolor='k')
axes[2].set_title('MinMax-scaled DTI')
plt.tight_layout()
plt.show()


## 4. Categorical Encoding


In [ ]:
# One-hot (pandas)
emp_dummies = pd.get_dummies(df['employment'], prefix='emp', dummy_na=True)
df = pd.concat([df, emp_dummies], axis=1)

# Simple target / mean encoding (educational – beware leakage in real CV)
emp_means = df.groupby('employment')['default'].mean()
df['emp_target_enc'] = df['employment'].map(emp_means)

# Ordinal example (arbitrary order for illustration)
ord_map = {'unemployed': 0, 'self-employed': 1, 'employed': 2}
df['emp_ordinal'] = df['employment'].map(ord_map)

print("Employment value counts:\n", df['employment'].value_counts(dropna=False))
print("\nTarget encoding map:\n", emp_means.round(3))
print(df[['employment','emp_target_enc','emp_ordinal']].head(8))


## 5. Interaction & Polynomial Features


In [ ]:
df['income_x_dti'] = df['income'] * df['dti']
df['age_x_hist']   = df['age'] * df['credit_history_years']
df['util_sq']      = df['utilization'].fillna(0) ** 2

print(df[['income_x_dti','age_x_hist','util_sq']].describe().round(2))


## 6. Missing-Value Indicators & Simple Imputation


In [ ]:
# Indicators
df['credit_limit_missing'] = df['credit_limit'].isna().astype(int)
df['employment_missing']   = df['employment'].isna().astype(int)

# Median / mode imputation
df['credit_limit_imp'] = df['credit_limit'].fillna(df['credit_limit'].median())
df['employment_imp']   = df['employment'].fillna(df['employment'].mode()[0])

# Recompute utilization with imputed limit
df['utilization'] = df['revolving_balance'] / df['credit_limit_imp']

print("Missing indicators sum:", df[['credit_limit_missing','employment_missing']].sum().to_dict())
print("Utilization after imputation – describe:")
print(df['utilization'].describe().round(3))


## 7. Assemble Feature Matrix


In [ ]:
feature_cols = [
    'log_income', 'dti_std', 'age_std', 'credit_history_years',
    'utilization', 'util_sq', 'income_x_dti', 'age_x_hist',
    'emp_target_enc', 'emp_ordinal',
    'credit_limit_missing', 'employment_missing'
]
# add one-hot columns that exist
feature_cols += [c for c in df.columns if c.startswith('emp_') and c not in feature_cols]

# keep only numeric and present columns
feature_cols = [c for c in feature_cols if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
X = df[feature_cols].fillna(0)
y = df['default']

print("Feature matrix shape:", X.shape)
print("Columns:", feature_cols)
print("\nCorrelation with default (top):")
print(X.corrwith(y).abs().sort_values(ascending=False).head(8).round(3))


## 8. Quick Downstream Check
Compare a logistic model trained on a few raw features versus the engineered set (AUC).


In [ ]:
if SKLEARN_AVAILABLE:
    raw_cols = ['income', 'debt', 'age', 'credit_history_years']
    X_raw = df[raw_cols].fillna(df[raw_cols].median())
    
    def quick_auc(X, y):
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.35, random_state=42, stratify=y)
        clf = LogisticRegression(max_iter=500, solver='lbfgs')
        clf.fit(Xtr, ytr)
        return roc_auc_score(yte, clf.predict_proba(Xte)[:,1])
    
    auc_raw = quick_auc(X_raw, y)
    auc_eng = quick_auc(X, y)
    print(f"AUC with raw features     : {auc_raw:.3f}")
    print(f"AUC with engineered feats : {auc_eng:.3f}")
    print("(Small sample – direction is more important than absolute numbers)")
else:
    print("sklearn not available – skipping model comparison. Correlations shown above.")


## 9. Alternate Implementation – scikit-learn style (sketch)
In production you would normally wrap the steps in `ColumnTransformer` + `Pipeline`.  
Below is a compact illustration of the same ideas.


In [ ]:
if SKLEARN_AVAILABLE:
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import OneHotEncoder
    
    numeric_features = ['income', 'debt', 'age', 'credit_history_years', 'revolving_balance', 'credit_limit']
    cat_features = ['employment']
    
    preproc = ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_features)
    ])
    
    pipe = Pipeline([
        ('prep', preproc),
        ('clf', LogisticRegression(max_iter=400))
    ])
    
    X_all = df[numeric_features + cat_features]
    Xtr, Xte, ytr, yte = train_test_split(X_all, y, test_size=0.35, random_state=42, stratify=y)
    pipe.fit(Xtr, ytr)
    auc_pipe = roc_auc_score(yte, pipe.predict_proba(Xte)[:,1])
    print(f"Pipeline AUC (raw + sklearn transforms): {auc_pipe:.3f}")
else:
    print("sklearn not available – alternate pipeline skipped.")


## 10. More Practice Results


In [ ]:
# Practice 1 – default rate by income quartile
print("Default rate by income quartile:")
print(df.groupby('income_bin', observed=True)['default'].mean().round(3))

# Practice 2 – high-utilization flag
df['high_util'] = (df['utilization'] > 0.7).astype(int)
print("\nHigh-utilization flag vs default:")
print(df.groupby('high_util')['default'].agg(['count','mean']).round(3))

# Practice 3 – visual of DTI bins
fig, ax = plt.subplots(figsize=(6, 3.5))
rates = df.groupby('dti_bin', observed=True)['default'].mean()
rates.plot(kind='bar', color='steelblue', edgecolor='k', ax=ax)
ax.set_ylabel('Default Rate')
ax.set_title('Default Rate by DTI Bin')
ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 11. Simulation Section – Change Engineering Choices


In [ ]:
# ========== SIMULATION PARAMETERS ==========
n_bins          = 4          # for income quantile bins
scale_method    = 'standard' # 'standard' or 'minmax'
include_inter   = True
# ===========================================

sim = df[['income','debt','age','default']].copy()
sim['dti'] = sim['debt'] / sim['income']

if scale_method == 'standard':
    sim['dti_scaled'] = (sim['dti'] - sim['dti'].mean()) / sim['dti'].std(ddof=0)
else:
    sim['dti_scaled'] = (sim['dti'] - sim['dti'].min()) / (sim['dti'].max() - sim['dti'].min() + 1e-9)

sim['income_bin'] = pd.qcut(sim['income'], q=n_bins, duplicates='drop')
if include_inter:
    sim['income_x_dti'] = sim['income'] * sim['dti']

print(f"Settings → bins={n_bins}, scale={scale_method}, interactions={include_inter}")
print("\nDefault rate by income bin:")
print(sim.groupby('income_bin', observed=True)['default'].agg(['count','mean']).round(3))
print("\nScaled DTI describe:")
print(sim['dti_scaled'].describe().round(3))

# quick visual
fig, ax = plt.subplots(figsize=(6, 3.5))
sim.boxplot(column='dti_scaled', by='income_bin', ax=ax)
ax.set_title('Scaled DTI by Income Bin')
ax.set_xlabel('Income Bin')
plt.suptitle('')
plt.tight_layout()
plt.show()


## 12. Audience Adaptation Notes
**Credit Risk / Data Science team** – show the full pipeline, correlation table, AUC lift, and the effect of different binning / scaling choices.  
**Credit Committee / business stakeholders** – emphasise the domain features they already understand (DTI, utilisation) and the simple bar chart of default rate by DTI bin. Avoid talking about “target encoding leakage” unless asked.


## Congratulations!
You now have a practical toolkit for feature engineering in a credit-risk setting:
- Domain ratios and logs that carry business meaning
- Scaling and binning that make features model-friendly
- Categorical encodings and interactions
- Missing-value indicators
- A quick way to check whether the engineering improved a simple downstream model

These steps almost always matter more than the final choice of algorithm.
